In [1]:
#pandas for spreadsheet processing
import pandas as pd
pd.set_option('display.max_columns', None) #display all columns

#numpy for mathematical functions
import numpy as np

#geopandas to process geospatial/gis data
import geopandas as gpd

#library for connecting to the geodatabase
from sqlalchemy import create_engine, text
from geoalchemy2 import Geometry

# User input

- All human input values will need to be input in by the user into the following cells. 
- The rest of the code will run without required human intervention.

## 1. Input Link to downloaded datasets

In [2]:
# National Identity
# To download original dataset go to - 
# https://www.ons.gov.uk/filters/377317a4-7dcc-4ac1-8c92-771f02792e66/dimensions
nat_identity_csv = r"N:\Geodatabase\Raw_Data\Census 2021\Passport\TS005-2021-3-filtered-2025-02-06T17_15_47Z.csv"

## 2. Input Link to LSOA 2021 shapefile
#### To download original dataset go to - 
https://geoportal.statistics.gov.uk/datasets/ons::lower-layer-super-output-areas-december-2021-boundaries-ew-bsc-v4-2/about

In [3]:
lsoa2021_shapefile = r"C:\Users\abhimanya.achara\OneDrive - priorpartners.com\Documents\UK DATA\2021 Census Data\Lower_Layer_Super_Output_Areas_(Dec_2021)_Boundaries_Full_Extent_(BFE)_EW\Lower_Layer_Super_Output_Areas_(Dec_2021)_Boundaries_Full_Extent_(BFE)_EW.shp"

## 3. Threshold to define dominant type
This defines the % threshold from the highest value of group of data columns to define which ones will be defined as dominant. 

In [4]:
threshold_value = 2.5

## 1. Import & Process Data

To download original dataset go to - 
https://www.nomisweb.co.uk/query/construct/summary.asp?mode=construct&version=0&dataset=2221

In [5]:
# Create Dataframe
lsoa_nat_identity_df = pd.read_csv(nat_identity_csv)
lsoa_nat_identity_df.head()

,Lower layer Super Output Areas Code,Lower layer Super Output Areas,National identity (17 categories) Code,National identity (17 categories),Observation
0,E01000001,City of London 001A,-8,Does not apply,0
1,E01000001,City of London 001A,1,British only identity,678
2,E01000001,City of London 001A,2,English only identity,104
3,E01000001,City of London 001A,3,English and British only identity,121
4,E01000001,City of London 001A,4,Welsh only identity,6


In [6]:
#create pivot table
lsoa_nat_identity_df_P = pd.pivot_table(lsoa_nat_identity_df, values='Observation', index=['Lower layer Super Output Areas Code'], columns=['National identity (17 categories)'], aggfunc=np.sum)
lsoa_nat_identity_df_P = lsoa_nat_identity_df_P.reset_index()

#drop columns
lsoa_nat_identity_df_P.drop(['Does not apply'], 1, inplace=True)

# Rename columns: lowercase, replace spaces with _, remove colons, and add '_count' suffix
lsoa_nat_identity_df_P.columns = (
    lsoa_nat_identity_df_P.columns
    .str.lower()                 # Convert to lowercase
    .str.replace(' ', '_')       # Replace spaces with underscores
    .str.replace(':', '')        # Remove colons
    .str.cat(['_count'] * len(lsoa_nat_identity_df_P.columns))  # Append '_count' to each column
)

#rename columns
lsoa_nat_identity_df_P.rename(columns = {'lower_layer_super_output_areas_code_count':'lsoa21cd'},inplace = True)
# Display the updated DataFrame

#create total population columns
lsoa_nat_identity_df_P['total_nat_identity_pop'] = lsoa_nat_identity_df_P.sum(axis=1,numeric_only=True)

lsoa_nat_identity_df_P.head()

C:\Users\abhimanya.achara\AppData\Local\Temp\ipykernel_10436\223091188.py:6: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only.
  lsoa_nat_identity_df_P.drop(['Does not apply'], 1, inplace=True)


National identity (17 categories),lsoa21cd,any_other_combination_of_only_uk_identities_count,british_only_identity_count,cornish_and_british_only_identity_count,cornish_only_identity_count,english_and_british_only_identity_count,english_only_identity_count,irish_and_at_least_one_uk_identity_count,irish_only_identity_count,northern_irish_and_british_only_identity_count,northern_irish_only_identity_count,other_identity_and_at_least_one_uk_identity_count,other_identity_only_count,scottish_and_british_only_identity_count,scottish_only_identity_count,welsh_and_british_only_identity_count,welsh_only_identity_count,total_nat_identity_pop
0,E01000001,2,678,3,0,121,104,5,18,0,5,115,394,9,8,5,6,1473
1,E01000002,7,603,0,1,114,104,11,26,0,5,118,373,7,11,3,6,1389
2,E01000003,5,745,0,2,118,149,9,18,1,10,139,388,5,13,3,7,1612
3,E01000005,0,671,0,0,57,54,0,6,0,1,32,276,1,3,0,0,1101
4,E01000006,0,1040,0,0,49,96,3,13,0,0,60,582,1,0,0,1,1845


## 2. Feature Engineering

In [7]:
# Create percentage columns for detailed ethnicity
total_nat_identity_pop = 'total_nat_identity_pop'
for col in lsoa_nat_identity_df_P.columns[1:-1]:
    perc_col = col.replace('_count', '_perc')
    lsoa_nat_identity_df_P[perc_col] = (lsoa_nat_identity_df_P[col] / lsoa_nat_identity_df_P[total_nat_identity_pop]) * 100

lsoa_nat_identity_df_P.head()

National identity (17 categories),lsoa21cd,any_other_combination_of_only_uk_identities_count,british_only_identity_count,cornish_and_british_only_identity_count,cornish_only_identity_count,english_and_british_only_identity_count,english_only_identity_count,irish_and_at_least_one_uk_identity_count,irish_only_identity_count,northern_irish_and_british_only_identity_count,northern_irish_only_identity_count,other_identity_and_at_least_one_uk_identity_count,other_identity_only_count,scottish_and_british_only_identity_count,scottish_only_identity_count,welsh_and_british_only_identity_count,welsh_only_identity_count,total_nat_identity_pop,any_other_combination_of_only_uk_identities_perc,british_only_identity_perc,cornish_and_british_only_identity_perc,cornish_only_identity_perc,english_and_british_only_identity_perc,english_only_identity_perc,irish_and_at_least_one_uk_identity_perc,irish_only_identity_perc,northern_irish_and_british_only_identity_perc,northern_irish_only_identity_perc,other_identity_and_at_least_one_uk_identity_perc,other_identity_only_perc,scottish_and_british_only_identity_perc,scottish_only_identity_perc,welsh_and_british_only_identity_perc,welsh_only_identity_perc
0,E01000001,2,678,3,0,121,104,5,18,0,5,115,394,9,8,5,6,1473,0.135777,46.028513,0.203666,0.000000,8.214528,7.060421,0.339443,1.221996,0.000000,0.339443,7.807196,26.748133,0.610998,0.543109,0.339443,0.407332
1,E01000002,7,603,0,1,114,104,11,26,0,5,118,373,7,11,3,6,1389,0.503960,43.412527,0.000000,0.071994,8.207343,7.487401,0.791937,1.871850,0.000000,0.359971,8.495320,26.853852,0.503960,0.791937,0.215983,0.431965
2,E01000003,5,745,0,2,118,149,9,18,1,10,139,388,5,13,3,7,1612,0.310174,46.215881,0.000000,0.124069,7.320099,9.243176,0.558313,1.116625,0.062035,0.620347,8.622829,24.069479,0.310174,0.806452,0.186104,0.434243
3,E01000005,0,671,0,0,57,54,0,6,0,1,32,276,1,3,0,0,1101,0.000000,60.944596,0.000000,0.000000,5.177112,4.904632,0.000000,0.544959,0.000000,0.090827,2.906449,25.068120,0.090827,0.272480,0.000000,0.000000
4,E01000006,0,1040,0,0,49,96,3,13,0,0,60,582,1,0,0,1,1845,0.000000,56.368564,0.000000,0.000000,2.655827,5.203252,0.162602,0.704607,0.000000,0.000000,3.252033,31.544715,0.054201,0.000000,0.000000,0.054201


## 4. Load GIS shapefile 

In [8]:
# Attempt to read from the server first, fallback to local path if it fails
lsoa2021boundary_df = gpd.read_file(lsoa2021_shapefile)
print("Shapefile loaded successfully from the server.")

# Display the first few rows
lsoa2021boundary_df.head()

Shapefile loaded successfully from the server.


,FID,LSOA21CD,LSOA21NM,Shape__Are,Shape__Len,GlobalID,geometry
0,1,E01000001,City of London 001A,129865.314476,2635.767993,4dd060c0-a44a-4c62-aca3-b0c88c0bf0d0,"POLYGON ((532151.537 181867.433, 532152.500 18..."
1,2,E01000002,City of London 001B,228419.782242,2707.816821,d0a47b5d-5649-4242-a8e7-462078975c1d,"POLYGON ((532634.497 181926.016, 532632.048 18..."
2,3,E01000003,City of London 001C,59054.204697,1224.573160,2904cc10-e994-4a6e-b11c-06be7fbd74d2,"POLYGON ((532153.703 182165.155, 532158.250 18..."
3,4,E01000005,City of London 001E,189577.709503,2275.805344,bfcf5958-f052-4388-8efd-75bb808d8b83,"POLYGON ((533619.062 181402.364, 533639.868 18..."
4,5,E01000006,Barking and Dagenham 016A,146536.995750,1966.092607,64a24862-79c1-4c8c-ab7b-d9cec1d3c0c6,"POLYGON ((545126.852 184310.838, 545145.213 18..."


## 5. GIS data management

### Remove Rename fields

In [9]:
#Drop and rename fields
lsoa2021boundary_df.drop(['Shape__Are', 'Shape__Len', 'GlobalID'], 1, inplace = True)
lsoa2021boundary_df.rename(columns = {'LSOA21CD':'lsoa21cd','LSOA21NM':'lsoa21nm'}, inplace = True)
lsoa2021boundary_df.head()

C:\Users\abhimanya.achara\AppData\Local\Temp\ipykernel_10436\2026090638.py:2: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only.
  lsoa2021boundary_df.drop(['Shape__Are', 'Shape__Len', 'GlobalID'], 1, inplace = True)


,FID,lsoa21cd,lsoa21nm,geometry
0,1,E01000001,City of London 001A,"POLYGON ((532151.537 181867.433, 532152.500 18..."
1,2,E01000002,City of London 001B,"POLYGON ((532634.497 181926.016, 532632.048 18..."
2,3,E01000003,City of London 001C,"POLYGON ((532153.703 182165.155, 532158.250 18..."
3,4,E01000005,City of London 001E,"POLYGON ((533619.062 181402.364, 533639.868 18..."
4,5,E01000006,Barking and Dagenham 016A,"POLYGON ((545126.852 184310.838, 545145.213 18..."


### Combine with boundary lookup table

#### LSOA21 to MSOA21 to LAD22

In [10]:
# Define the file path for msoa11 to msoa21 lookup
file_path = r"N:\Geodatabase\Raw_Data\Look up tables\OA21_LSOA21_MSOA21_LAD22_EW_LU.csv"

# Read the Excel file
lsoalookup_df = pd.read_csv(file_path)  # Reads the first sheet by default

#drop oa column
lsoalookup_df.drop(['oa21cd','lsoa21nm'],1,inplace = True)

#remove duplicates
lsoalookup_df = lsoalookup_df.drop_duplicates(subset='lsoa21cd', keep='first')

# Display the first few rows
lsoalookup_df.head()

C:\Users\abhimanya.achara\AppData\Local\Temp\ipykernel_10436\1268765203.py:8: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only.
  lsoalookup_df.drop(['oa21cd','lsoa21nm'],1,inplace = True)


,lsoa21cd,msoa21cd,msoa21nm,lad22cd,lad22nm
0,E01000001,E02000001,City of London 001,E09000001,City of London
4,E01000003,E02000001,City of London 001,E09000001,City of London
6,E01000002,E02000001,City of London 001,E09000001,City of London
12,E01032739,E02000001,City of London 001,E09000001,City of London
13,E01032740,E02000001,City of London 001,E09000001,City of London


In [11]:
lsoa2021boundary_df = pd.merge(lsoa2021boundary_df, lsoalookup_df, how = 'left', on = 'lsoa21cd')
lsoa2021boundary_df.head()

,FID,lsoa21cd,lsoa21nm,geometry,msoa21cd,msoa21nm,lad22cd,lad22nm
0,1,E01000001,City of London 001A,"POLYGON ((532151.537 181867.433, 532152.500 18...",E02000001,City of London 001,E09000001,City of London
1,2,E01000002,City of London 001B,"POLYGON ((532634.497 181926.016, 532632.048 18...",E02000001,City of London 001,E09000001,City of London
2,3,E01000003,City of London 001C,"POLYGON ((532153.703 182165.155, 532158.250 18...",E02000001,City of London 001,E09000001,City of London
3,4,E01000005,City of London 001E,"POLYGON ((533619.062 181402.364, 533639.868 18...",E02000001,City of London 001,E09000001,City of London
4,5,E01000006,Barking and Dagenham 016A,"POLYGON ((545126.852 184310.838, 545145.213 18...",E02000017,Barking and Dagenham 016,E09000002,Barking and Dagenham


#### LSOA21 to WARD22

In [12]:
# Define the file path for msoa21 to ward lookup
file_path = r"N:\Geodatabase\Raw_Data\Look up tables\Lower_Layer_Super_Output_Area_(2021)_to_Ward_(2022)_to_LAD_(2022)_Lookup_in_England_and_Wales_v3.csv"

# Read the Excel file
lsoawardlookup_df = pd.read_csv(file_path)  # Reads the first sheet by default

#drop unwanted fields
lsoawardlookup_df.drop(['LSOA21NM','LSOA21NMW','WD22NMW','LAD22NMW','ObjectId','LAD22CD','LAD22NM'],1,inplace = True)

# Rename fields
lsoawardlookup_df.rename(
    columns={
        'LSOA21CD': 'lsoa21cd',
        'WD22CD': 'wd22cd',
        'WD22NM': 'wd22nm',
        }, 
    inplace=True
)

# Display the first few rows
lsoawardlookup_df.head()

C:\Users\abhimanya.achara\AppData\Local\Temp\ipykernel_10436\1707502962.py:8: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only.
  lsoawardlookup_df.drop(['LSOA21NM','LSOA21NMW','WD22NMW','LAD22NMW','ObjectId','LAD22CD','LAD22NM'],1,inplace = True)


,lsoa21cd,wd22cd,wd22nm
0,E01004766,E05000650,Astley Bridge
1,E01005347,E05000722,Chadderton South
2,E01004890,E05000661,Horwich North East
3,E01004770,E05000650,Astley Bridge
4,E01004888,E05000661,Horwich North East


In [13]:
lsoa2021boundary_df = pd.merge(lsoa2021boundary_df, lsoawardlookup_df, how = 'inner', on = 'lsoa21cd')
lsoa2021boundary_df.head()

,FID,lsoa21cd,lsoa21nm,geometry,msoa21cd,msoa21nm,lad22cd,lad22nm,wd22cd,wd22nm
0,1,E01000001,City of London 001A,"POLYGON ((532151.537 181867.433, 532152.500 18...",E02000001,City of London 001,E09000001,City of London,E05009288,Aldersgate
1,2,E01000002,City of London 001B,"POLYGON ((532634.497 181926.016, 532632.048 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate
2,3,E01000003,City of London 001C,"POLYGON ((532153.703 182165.155, 532158.250 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate
3,4,E01000005,City of London 001E,"POLYGON ((533619.062 181402.364, 533639.868 18...",E02000001,City of London 001,E09000001,City of London,E05009308,Portsoken
4,5,E01000006,Barking and Dagenham 016A,"POLYGON ((545126.852 184310.838, 545145.213 18...",E02000017,Barking and Dagenham 016,E09000002,Barking and Dagenham,E05014066,Northbury


#### LAD22 to REGION22

In [14]:
# Define the file path for msoa21 to ward lookup
file_path = r"N:\Geodatabase\Raw_Data\Look up tables\Local_Authority_District_to_Region_(December_2022)_Lookup_in_England.csv"

# Read the Excel file
ladregionlookup_df = pd.read_csv(file_path)  # Reads the first sheet by default

#drop unwanted fields
ladregionlookup_df.drop(['LAD22NM','ObjectId'],1,inplace = True)

# Rename fields
ladregionlookup_df.rename(
    columns={
        'LAD22CD': 'lad22cd',
        'RGN22CD': 'rgn22cd',
        'RGN22NM': 'rgn22nm'
       }, 
    inplace=True
)

# Display the first few rows
ladregionlookup_df.head()

C:\Users\abhimanya.achara\AppData\Local\Temp\ipykernel_10436\4188529333.py:8: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only.
  ladregionlookup_df.drop(['LAD22NM','ObjectId'],1,inplace = True)


,lad22cd,rgn22cd,rgn22nm
0,E06000001,E12000001,North East
1,E06000002,E12000001,North East
2,E06000003,E12000001,North East
3,E06000004,E12000001,North East
4,E06000005,E12000001,North East


In [15]:
lsoa2021boundary_df = pd.merge(lsoa2021boundary_df, ladregionlookup_df, how = 'left', on = 'lad22cd')
lsoa2021boundary_df.head()

,FID,lsoa21cd,lsoa21nm,geometry,msoa21cd,msoa21nm,lad22cd,lad22nm,wd22cd,wd22nm,rgn22cd,rgn22nm
0,1,E01000001,City of London 001A,"POLYGON ((532151.537 181867.433, 532152.500 18...",E02000001,City of London 001,E09000001,City of London,E05009288,Aldersgate,E12000007,London
1,2,E01000002,City of London 001B,"POLYGON ((532634.497 181926.016, 532632.048 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate,E12000007,London
2,3,E01000003,City of London 001C,"POLYGON ((532153.703 182165.155, 532158.250 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate,E12000007,London
3,4,E01000005,City of London 001E,"POLYGON ((533619.062 181402.364, 533639.868 18...",E02000001,City of London 001,E09000001,City of London,E05009308,Portsoken,E12000007,London
4,5,E01000006,Barking and Dagenham 016A,"POLYGON ((545126.852 184310.838, 545145.213 18...",E02000017,Barking and Dagenham 016,E09000002,Barking and Dagenham,E05014066,Northbury,E12000007,London


### Add data management fields

In [16]:
# Add new data management fields with specified values
lsoa2021boundary_df['data_source'] = "Census 2021 (ONS Nomis - Official Census and Labour Market Statistics)"
lsoa2021boundary_df['data_resolution'] = "LSOA 2021"
lsoa2021boundary_df['data_time_period'] = pd.to_datetime("2021-02-21")  # Convert to date format
lsoa2021boundary_df['data_web_link'] = "https://www.nomisweb.co.uk/"
lsoa2021boundary_df.head()

,FID,lsoa21cd,lsoa21nm,geometry,msoa21cd,msoa21nm,lad22cd,lad22nm,wd22cd,wd22nm,rgn22cd,rgn22nm,data_source,data_resolution,data_time_period,data_web_link
0,1,E01000001,City of London 001A,"POLYGON ((532151.537 181867.433, 532152.500 18...",E02000001,City of London 001,E09000001,City of London,E05009288,Aldersgate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/
1,2,E01000002,City of London 001B,"POLYGON ((532634.497 181926.016, 532632.048 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/
2,3,E01000003,City of London 001C,"POLYGON ((532153.703 182165.155, 532158.250 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/
3,4,E01000005,City of London 001E,"POLYGON ((533619.062 181402.364, 533639.868 18...",E02000001,City of London 001,E09000001,City of London,E05009308,Portsoken,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/
4,5,E01000006,Barking and Dagenham 016A,"POLYGON ((545126.852 184310.838, 545145.213 18...",E02000017,Barking and Dagenham 016,E09000002,Barking and Dagenham,E05014066,Northbury,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/


### Update Area

In [17]:
#Update Area in Hectares
# Ensure the dataframe has a valid geometry column
if 'geometry' in lsoa2021boundary_df.columns:
    # Calculate the area in square meters and convert to hectares (1 hectare = 10,000 m²)
    lsoa2021boundary_df['area_ha'] = lsoa2021boundary_df.geometry.area / 10_000

    # Print confirmation message
    print("Field 'area_ha' has been added and updated successfully.")
else:
    print("Error: No 'geometry' column found in lsoa2021boundary_df. Ensure it's a GeoDataFrame with valid geometries.")
    
lsoa2021boundary_df.head()

Field 'area_ha' has been added and updated successfully.


,FID,lsoa21cd,lsoa21nm,geometry,msoa21cd,msoa21nm,lad22cd,lad22nm,wd22cd,wd22nm,rgn22cd,rgn22nm,data_source,data_resolution,data_time_period,data_web_link,area_ha
0,1,E01000001,City of London 001A,"POLYGON ((532151.537 181867.433, 532152.500 18...",E02000001,City of London 001,E09000001,City of London,E05009288,Aldersgate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,12.986531
1,2,E01000002,City of London 001B,"POLYGON ((532634.497 181926.016, 532632.048 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,22.841978
2,3,E01000003,City of London 001C,"POLYGON ((532153.703 182165.155, 532158.250 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,5.905420
3,4,E01000005,City of London 001E,"POLYGON ((533619.062 181402.364, 533639.868 18...",E02000001,City of London 001,E09000001,City of London,E05009308,Portsoken,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,18.957771
4,5,E01000006,Barking and Dagenham 016A,"POLYGON ((545126.852 184310.838, 545145.213 18...",E02000017,Barking and Dagenham 016,E09000002,Barking and Dagenham,E05014066,Northbury,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,14.653700


# 7. Combine Geometry and data

In [18]:
census2021_nat_identity_lsoa2021_gdb_df = pd.merge(lsoa2021boundary_df,lsoa_nat_identity_df_P, how = 'left', on='lsoa21cd')
census2021_nat_identity_lsoa2021_gdb_df.head()

,FID,lsoa21cd,lsoa21nm,geometry,msoa21cd,msoa21nm,lad22cd,lad22nm,wd22cd,wd22nm,rgn22cd,rgn22nm,data_source,data_resolution,data_time_period,data_web_link,area_ha,any_other_combination_of_only_uk_identities_count,british_only_identity_count,cornish_and_british_only_identity_count,cornish_only_identity_count,english_and_british_only_identity_count,english_only_identity_count,irish_and_at_least_one_uk_identity_count,irish_only_identity_count,northern_irish_and_british_only_identity_count,northern_irish_only_identity_count,other_identity_and_at_least_one_uk_identity_count,other_identity_only_count,scottish_and_british_only_identity_count,scottish_only_identity_count,welsh_and_british_only_identity_count,welsh_only_identity_count,total_nat_identity_pop,any_other_combination_of_only_uk_identities_perc,british_only_identity_perc,cornish_and_british_only_identity_perc,cornish_only_identity_perc,english_and_british_only_identity_perc,english_only_identity_perc,irish_and_at_least_one_uk_identity_perc,irish_only_identity_perc,northern_irish_and_british_only_identity_perc,northern_irish_only_identity_perc,other_identity_and_at_least_one_uk_identity_perc,other_identity_only_perc,scottish_and_british_only_identity_perc,scottish_only_identity_perc,welsh_and_british_only_identity_perc,welsh_only_identity_perc
0,1,E01000001,City of London 001A,"POLYGON ((532151.537 181867.433, 532152.500 18...",E02000001,City of London 001,E09000001,City of London,E05009288,Aldersgate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,12.986531,2,678,3,0,121,104,5,18,0,5,115,394,9,8,5,6,1473,0.135777,46.028513,0.203666,0.000000,8.214528,7.060421,0.339443,1.221996,0.000000,0.339443,7.807196,26.748133,0.610998,0.543109,0.339443,0.407332
1,2,E01000002,City of London 001B,"POLYGON ((532634.497 181926.016, 532632.048 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,22.841978,7,603,0,1,114,104,11,26,0,5,118,373,7,11,3,6,1389,0.503960,43.412527,0.000000,0.071994,8.207343,7.487401,0.791937,1.871850,0.000000,0.359971,8.495320,26.853852,0.503960,0.791937,0.215983,0.431965
2,3,E01000003,City of London 001C,"POLYGON ((532153.703 182165.155, 532158.250 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,5.905420,5,745,0,2,118,149,9,18,1,10,139,388,5,13,3,7,1612,0.310174,46.215881,0.000000,0.124069,7.320099,9.243176,0.558313,1.116625,0.062035,0.620347,8.622829,24.069479,0.310174,0.806452,0.186104,0.434243
3,4,E01000005,City of London 001E,"POLYGON ((533619.062 181402.364, 533639.868 18...",E02000001,City of London 001,E09000001,City of London,E05009308,Portsoken,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,18.957771,0,671,0,0,57,54,0,6,0,1,32,276,1,3,0,0,1101,0.000000,60.944596,0.000000,0.000000,5.177112,4.904632,0.000000,0.544959,0.000000,0.090827,2.906449,25.068120,0.090827,0.272480,0.000000,0.000000
4,5,E01000006,Barking and Dagenham 016A,"POLYGON ((545126.852 184310.838, 545145.213 18...",E02000017,Barking and Dagenham 016,E09000002,Barking and Dagenham,E05014066,Northbury,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,14.653700,0,1040,0,0,49,96,3,13,0,0,60,582,1,0,0,1,1845,0.000000,56.368564,0.000000,0.000000,2.655827,5.203252,0.162602,0.704607,0.000000,0.000000,3.252033,31.544715,0.054201,0.000000,0.000000,0.054201


# 8. Final tweaks

In [20]:
# Reorder columns in the combined dataframe

final_column_order = ['FID',
 'lsoa21cd',
 'lsoa21nm',
 'geometry',
 'msoa21cd',
 'msoa21nm',
 'wd22cd',
 'wd22nm',
 'lad22cd',
 'lad22nm',
 'rgn22cd',
 'rgn22nm',
 'data_source',
 'data_resolution',
 'data_time_period',
 'data_web_link',
 'area_ha',
 'any_other_combination_of_only_uk_identities_count',
 'british_only_identity_count',
 'cornish_and_british_only_identity_count',
 'cornish_only_identity_count',
 'english_and_british_only_identity_count',
 'english_only_identity_count',
 'irish_and_at_least_one_uk_identity_count',
 'irish_only_identity_count',
 'northern_irish_and_british_only_identity_count',
 'northern_irish_only_identity_count',
 'other_identity_and_at_least_one_uk_identity_count',
 'other_identity_only_count',
 'scottish_and_british_only_identity_count',
 'scottish_only_identity_count',
 'welsh_and_british_only_identity_count',
 'welsh_only_identity_count',
 'total_nat_identity_pop',
 'any_other_combination_of_only_uk_identities_perc',
 'british_only_identity_perc',
 'cornish_and_british_only_identity_perc',
 'cornish_only_identity_perc',
 'english_and_british_only_identity_perc',
 'english_only_identity_perc',
 'irish_and_at_least_one_uk_identity_perc',
 'irish_only_identity_perc',
 'northern_irish_and_british_only_identity_perc',
 'northern_irish_only_identity_perc',
 'other_identity_and_at_least_one_uk_identity_perc',
 'other_identity_only_perc',
 'scottish_and_british_only_identity_perc',
 'scottish_only_identity_perc',
 'welsh_and_british_only_identity_perc',
 'welsh_only_identity_perc']

census2021_nat_identity_lsoa2021_gdb_df = census2021_nat_identity_lsoa2021_gdb_df[final_column_order]

census2021_nat_identity_lsoa2021_gdb_df.head()

,FID,lsoa21cd,lsoa21nm,geometry,msoa21cd,msoa21nm,wd22cd,wd22nm,lad22cd,lad22nm,rgn22cd,rgn22nm,data_source,data_resolution,data_time_period,data_web_link,area_ha,any_other_combination_of_only_uk_identities_count,british_only_identity_count,cornish_and_british_only_identity_count,cornish_only_identity_count,english_and_british_only_identity_count,english_only_identity_count,irish_and_at_least_one_uk_identity_count,irish_only_identity_count,northern_irish_and_british_only_identity_count,northern_irish_only_identity_count,other_identity_and_at_least_one_uk_identity_count,other_identity_only_count,scottish_and_british_only_identity_count,scottish_only_identity_count,welsh_and_british_only_identity_count,welsh_only_identity_count,total_nat_identity_pop,any_other_combination_of_only_uk_identities_perc,british_only_identity_perc,cornish_and_british_only_identity_perc,cornish_only_identity_perc,english_and_british_only_identity_perc,english_only_identity_perc,irish_and_at_least_one_uk_identity_perc,irish_only_identity_perc,northern_irish_and_british_only_identity_perc,northern_irish_only_identity_perc,other_identity_and_at_least_one_uk_identity_perc,other_identity_only_perc,scottish_and_british_only_identity_perc,scottish_only_identity_perc,welsh_and_british_only_identity_perc,welsh_only_identity_perc
0,1,E01000001,City of London 001A,"POLYGON ((532151.537 181867.433, 532152.500 18...",E02000001,City of London 001,E05009288,Aldersgate,E09000001,City of London,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,12.986531,2,678,3,0,121,104,5,18,0,5,115,394,9,8,5,6,1473,0.135777,46.028513,0.203666,0.000000,8.214528,7.060421,0.339443,1.221996,0.000000,0.339443,7.807196,26.748133,0.610998,0.543109,0.339443,0.407332
1,2,E01000002,City of London 001B,"POLYGON ((532634.497 181926.016, 532632.048 18...",E02000001,City of London 001,E05009302,Cripplegate,E09000001,City of London,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,22.841978,7,603,0,1,114,104,11,26,0,5,118,373,7,11,3,6,1389,0.503960,43.412527,0.000000,0.071994,8.207343,7.487401,0.791937,1.871850,0.000000,0.359971,8.495320,26.853852,0.503960,0.791937,0.215983,0.431965
2,3,E01000003,City of London 001C,"POLYGON ((532153.703 182165.155, 532158.250 18...",E02000001,City of London 001,E05009302,Cripplegate,E09000001,City of London,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,5.905420,5,745,0,2,118,149,9,18,1,10,139,388,5,13,3,7,1612,0.310174,46.215881,0.000000,0.124069,7.320099,9.243176,0.558313,1.116625,0.062035,0.620347,8.622829,24.069479,0.310174,0.806452,0.186104,0.434243
3,4,E01000005,City of London 001E,"POLYGON ((533619.062 181402.364, 533639.868 18...",E02000001,City of London 001,E05009308,Portsoken,E09000001,City of London,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,18.957771,0,671,0,0,57,54,0,6,0,1,32,276,1,3,0,0,1101,0.000000,60.944596,0.000000,0.000000,5.177112,4.904632,0.000000,0.544959,0.000000,0.090827,2.906449,25.068120,0.090827,0.272480,0.000000,0.000000
4,5,E01000006,Barking and Dagenham 016A,"POLYGON ((545126.852 184310.838, 545145.213 18...",E02000017,Barking and Dagenham 016,E05014066,Northbury,E09000002,Barking and Dagenham,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,14.653700,0,1040,0,0,49,96,3,13,0,0,60,582,1,0,0,1,1845,0.000000,56.368564,0.000000,0.000000,2.655827,5.203252,0.162602,0.704607,0.000000,0.000000,3.252033,31.544715,0.054201,0.000000,0.000000,0.054201


# 8. Create dominant field

In [23]:
def determine_dominant_group(row):
    columns = [
 'any_other_combination_of_only_uk_identities_perc',
 'british_only_identity_perc',
 'cornish_and_british_only_identity_perc',
 'cornish_only_identity_perc',
 'english_and_british_only_identity_perc',
 'english_only_identity_perc',
 'irish_and_at_least_one_uk_identity_perc',
 'irish_only_identity_perc',
 'northern_irish_and_british_only_identity_perc',
 'northern_irish_only_identity_perc',
 'other_identity_and_at_least_one_uk_identity_perc',
 'other_identity_only_perc',
 'scottish_and_british_only_identity_perc',
 'scottish_only_identity_perc',
 'welsh_and_british_only_identity_perc',
 'welsh_only_identity_perc'
]
    
    max_value = max(row[col] for col in columns)
    threshold = max_value - (threshold_value)
    
    dominant_groups = [col.replace('_perc', '') for col in columns if row[col] >= threshold]
    
    return ', '.join(dominant_groups)

census2021_nat_identity_lsoa2021_gdb_df['dominant_national_identity_group'] = census2021_nat_identity_lsoa2021_gdb_df.apply(determine_dominant_group, axis=1)

# Display the first few rows of the updated dataframe
census2021_nat_identity_lsoa2021_gdb_df.head()

,FID,lsoa21cd,lsoa21nm,geometry,msoa21cd,msoa21nm,wd22cd,wd22nm,lad22cd,lad22nm,rgn22cd,rgn22nm,data_source,data_resolution,data_time_period,data_web_link,area_ha,any_other_combination_of_only_uk_identities_count,british_only_identity_count,cornish_and_british_only_identity_count,cornish_only_identity_count,english_and_british_only_identity_count,english_only_identity_count,irish_and_at_least_one_uk_identity_count,irish_only_identity_count,northern_irish_and_british_only_identity_count,northern_irish_only_identity_count,other_identity_and_at_least_one_uk_identity_count,other_identity_only_count,scottish_and_british_only_identity_count,scottish_only_identity_count,welsh_and_british_only_identity_count,welsh_only_identity_count,total_nat_identity_pop,any_other_combination_of_only_uk_identities_perc,british_only_identity_perc,cornish_and_british_only_identity_perc,cornish_only_identity_perc,english_and_british_only_identity_perc,english_only_identity_perc,irish_and_at_least_one_uk_identity_perc,irish_only_identity_perc,northern_irish_and_british_only_identity_perc,northern_irish_only_identity_perc,other_identity_and_at_least_one_uk_identity_perc,other_identity_only_perc,scottish_and_british_only_identity_perc,scottish_only_identity_perc,welsh_and_british_only_identity_perc,welsh_only_identity_perc,dominant_national_identity_group
0,1,E01000001,City of London 001A,"POLYGON ((532151.537 181867.433, 532152.500 18...",E02000001,City of London 001,E05009288,Aldersgate,E09000001,City of London,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,12.986531,2,678,3,0,121,104,5,18,0,5,115,394,9,8,5,6,1473,0.135777,46.028513,0.203666,0.000000,8.214528,7.060421,0.339443,1.221996,0.000000,0.339443,7.807196,26.748133,0.610998,0.543109,0.339443,0.407332,british_only_identity
1,2,E01000002,City of London 001B,"POLYGON ((532634.497 181926.016, 532632.048 18...",E02000001,City of London 001,E05009302,Cripplegate,E09000001,City of London,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,22.841978,7,603,0,1,114,104,11,26,0,5,118,373,7,11,3,6,1389,0.503960,43.412527,0.000000,0.071994,8.207343,7.487401,0.791937,1.871850,0.000000,0.359971,8.495320,26.853852,0.503960,0.791937,0.215983,0.431965,british_only_identity
2,3,E01000003,City of London 001C,"POLYGON ((532153.703 182165.155, 532158.250 18...",E02000001,City of London 001,E05009302,Cripplegate,E09000001,City of London,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,5.905420,5,745,0,2,118,149,9,18,1,10,139,388,5,13,3,7,1612,0.310174,46.215881,0.000000,0.124069,7.320099,9.243176,0.558313,1.116625,0.062035,0.620347,8.622829,24.069479,0.310174,0.806452,0.186104,0.434243,british_only_identity
3,4,E01000005,City of London 001E,"POLYGON ((533619.062 181402.364, 533639.868 18...",E02000001,City of London 001,E05009308,Portsoken,E09000001,City of London,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,18.957771,0,671,0,0,57,54,0,6,0,1,32,276,1,3,0,0,1101,0.000000,60.944596,0.000000,0.000000,5.177112,4.904632,0.000000,0.544959,0.000000,0.090827,2.906449,25.068120,0.090827,0.272480,0.000000,0.000000,british_only_identity
4,5,E01000006,Barking and Dagenham 016A,"POLYGON ((545126.852 184310.838, 545145.213 18...",E02000017,Barking and Dagenham 016,E05014066,Northbury,E09000002,Barking and Dagenham,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,14.653700,0,1040,0,0,49,96,3,13,0,0,60,582,1,0,0,1,1845,0.000000,56.368564,0.000000,0.000000,2.655827,5.203252,0.162602,0.704607,0.000000,0.000000,3.252033,31.544715,0.054201,0.000000,0.000000,0.054201,british_only_identity


# 9. Upload to geodatabase

In [24]:
# Define database connection parameters
db_host = "PRIORPSRV03"
db_name = "gis"
db_port = "5432"
db_schema = "uk_new"
table_name = "census2021_lsoa2021_national_identity"  # Desired table name
primary_key_column = "lsoa21cd"  # Define the primary key column (change based on your dataset)
geometry_column = "geometry"  # Default geometry column
# Create the database connection string (Windows Authentication - Trusted Connection)
conn_str = f"postgresql+psycopg2://@{db_host}:{db_port}/{db_name}?sslmode=disable"

# Create a SQLAlchemy engine
engine = create_engine(conn_str)

In [25]:
# Ensure the GeoDataFrame has a valid CRS before writing
if census2021_nat_identity_lsoa2021_gdb_df.crs is None:
    print("Warning: GeoDataFrame has no CRS. Setting default to EPSG:27700 (British National Grid).")
    census2021_nat_identity_lsoa2021_gdb_df.set_crs(epsg=27700, inplace=True)

In [26]:
# Publish the GeoDataFrame to PostGIS
census2021_nat_identity_lsoa2021_gdb_df.to_postgis(
    name=table_name,
    con=engine,
    schema=db_schema,
    if_exists="replace",
    index=False,
    dtype={'geometry': 'MULTIPOLYGON'}  # Ensure geometry is stored as MULTIPOLYGON
)

print(f"Data successfully uploaded to PostGIS: {db_schema}.{table_name}")

# Connect to the database to modify table structure
with engine.connect() as conn:
    # Set Primary Key (if it doesn't exist already)
    alter_pk_query = text(f"""
        ALTER TABLE {db_schema}.{table_name}
        ADD CONSTRAINT {table_name}_pkey PRIMARY KEY ({primary_key_column});
    """)
    
    # Create Spatial Index
    create_spatial_index_query = text(f"""
        CREATE INDEX {table_name}_geom_idx
        ON {db_schema}.{table_name}
        USING GIST ({geometry_column});
    """)

    try:
        conn.execute(alter_pk_query)  # Add Primary Key
        print(f"Primary key set on column: {primary_key_column}")
    except Exception as e:
        print(f"Could not set primary key. It may already exist. Error: {e}")

    try:
        conn.execute(create_spatial_index_query)  # Add Spatial Index
        print(f"Spatial index created for geometry column: {geometry_column}")
    except Exception as e:
        print(f"Could not create spatial index. It may already exist. Error: {e}")

print(f"GeoDataFrame successfully published to PostGIS with Primary Key and Spatial Index: {db_schema}.{table_name}")

Data successfully uploaded to PostGIS: uk_new.census2021_lsoa2021_national_identity
Primary key set on column: lsoa21cd
Spatial index created for geometry column: geometry
GeoDataFrame successfully published to PostGIS with Primary Key and Spatial Index: uk_new.census2021_lsoa2021_national_identity
